# Mini-Challenge 3 — Refactor a messy notebook into something exam-ready
### Saint Mary's · Friday lunch break

> *"This works, but the board can't read it. Make it presentable."*

**Time:** 15 min · **Mode:** solo · **Goal:** see what 'clean' looks like in practice.

---

Below you'll find a notebook cell that "works". Refactor it according to the clean-notebook checklist:

1. Markdown title + one-line motivation
2. Setup cell (imports + data) — separate from analysis
3. Helper functions defined together
4. Each code cell has ONE job
5. Variables named for meaning (no `xx`, `tmp`, `z`)
6. Plot has title + labels
7. No commented-out dead code


# Solution — Mini-Challenge 3 — Margin per Ward at Saint Mary's

*Q1 2026. We compute the total margin (DRG reimbursement − actual cost) by ward to identify which wards run profitably and which need attention.*

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

HERE = Path.cwd()
DATA = next((p / "data" for p in [HERE, *HERE.parents] if (p / "data").exists()), None)
assert DATA is not None, "data/ folder not found"

enc = pd.read_csv(DATA / "encounters.csv", parse_dates=["admission_date","discharge_date"])
drg = pd.read_csv(DATA / "drg_catalog.csv")
patients = pd.read_csv(DATA / "patients.csv")

## Helper function

In [ ]:
def margin_by_ward(encounters_df, drg_df):
    """Return a Series of total margin (EUR) per ward, sorted ascending."""
    enriched = encounters_df.merge(drg_df, on="drg_code", how="left")
    enriched["margin_eur"] = enriched["base_reimbursement_eur"] - enriched["actual_cost_eur"]
    return enriched.groupby("ward")["margin_eur"].sum().sort_values()

## Result

In [ ]:
ward_margin = margin_by_ward(enc, drg)
ward_margin.round(2)

## Visualisation

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#c0392b" if v < 0 else "#2e8b57" for v in ward_margin.values]
ward_margin.plot(kind="barh", ax=ax, color=colors)
ax.set_title("Total margin by ward · Saint Mary's Q1 2026")
ax.set_xlabel("Total margin (EUR)")
ax.set_ylabel("Ward")
ax.axvline(0, color="black", linewidth=1)
plt.tight_layout()
plt.show()

## Reflection

The messy version hid the logic in one cell with cryptic variable names (`xx`, `tmp`, `z`, `m`), commented-out dead code, and a chart with no title or labels. The clean version separates **what** (the question) from **how** (the function `margin_by_ward`) and **the picture** — so the next reader can scan in 30 seconds. One improvement: I would add a docstring to the function specifying what columns are required in the input DataFrames.